# Phases 7b and 7c: a new guard, and reports from 2025-2026

Write-ups: [`docs/GUARD_7B.md`](../docs/GUARD_7B.md), [`docs/DATA_2025_2026.md`](../docs/DATA_2025_2026.md).

**Key ideas**
- **Paired test:** the same answer with and without one planted unsafe sentence.
- **Fingerprint:** a number that changes if any row of a table changes.
- **Scope:** the same fixed lookups over 2022-2024 (default) or 2022-2026.

In [1]:
import json
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
load = lambda p: json.loads((ROOT / 'results' / p).read_text(encoding='utf-8'))
G = load('phase7b/test_grades.json')
print('7b labels by Claude Opus, no human check; passes:', G['passes'])
pd.Series({k: G[k] for k in ('caught', 'missed', 'clean_withheld', 'new_withheld', 'plants_labelled_unsafe')})

7b labels by Claude Opus, no human check; passes: True


caught                         38/40
missed                    [P07, P35]
clean_withheld                    []
new_withheld                   [N08]
plants_labelled_unsafe           8/8
dtype: object

## 7b: why we did these tasks

| Task | Why we did it | What we learned | What it changes next |
|---|---|---|---|
| Froze the test before the prompt | The earlier prompt had leaked into its test | No shared four-word runs (tested in CI) | Same rule for any prompt test |
| Made the guard quote its reason | Phase 7 could not show why answers were withheld | v1's quotes were harmless sentences; asking for a reason and verdict fixed it in development | Every withheld answer can be explained |
| Paired clean and planted answers | To know the sentence, not the topic, triggers the guard | 38/40 caught, 0/40 clean withheld | Use pairs for new guard tests |
| Labels committed before verdicts | So labels cannot follow the guard | Kept in order, but the labeller is an AI | A human check is still the missing piece |
| Read-only lookups | The login could delete reports | The database now refuses writes during lookups | Phase 8's app uses the same path |


## 7c: independent checks

In [2]:
V = load('phase7c/verify_phase7c.json')
print(V['passed'])
pd.DataFrame([(c['id'], c['check'], c['pass']) for c in V['checks']], columns=['id', 'check', 'pass'])

9/9


,id,check,pass
0,D1,loaded + quarantined = raw lines; recalls = op...,True
1,D2,every row rebuilt from raw,True
2,D3,"core, analytics and memory unchanged",True
3,D4,2022-2024 part equals core,True
4,D5,"one mode per report, recomputed; quarter count...",True
5,D6,default scope identical; scope 'all' equals se...,True
6,D7,one vector per report; old vectors identical; ...,True
7,D8,no contact or demographic columns; sampled con...,True
8,D9,"planted null term, duplicate and late date han...",True


## 7c: Haiku on 2022-2026

In [3]:
M = load('phase7c/model_check.json')
print(M['correct'], 'bar', M['bar'], '$', M['usd'])
pd.DataFrame(M['rows'])[['id', 'category', 'correct', 'steps', 'usd']]

20/20 bar 18 $ 0.02559


,id,category,correct,steps,usd
0,C01,count,True,2,0.000974
1,C02,count,True,2,0.001094
2,C03,count,True,2,0.000958
3,C04,count,True,2,0.000945
4,C05,count,True,2,0.000981
5,C06,trend,True,3,0.001181
6,C07,trend,True,3,0.001207
7,C08,trend,True,3,0.001206
8,C09,mode,True,2,0.001179
9,C10,mode,True,2,0.001173


## 7c: why we did these tasks

| Task | Why we did it | What we learned | What it changes next |
|---|---|---|---|
| Fingerprinted the old tables | To prove nothing finished was disturbed | 18 tables identical afterwards | Same for any later load |
| Kept the old scope as default | Recorded answers must still replay | 233 of 233 recorded lookups identical | Phase 8 can offer both scopes |
| Asked why reports were rejected | 3,663 was far from the usual 1 | FDA publishes the text months after the rest | Recent counts include them; text search says how many it cannot see |
| Rebuilt every row from raw | A load can be wrong without failing | 0 differences in 233,264 reports | Runs again after any reload |
| Questions that need paging | To test the new access, not just old lookups | Haiku paged correctly in all four | Harder analysis questions would be the next test |
